In [3]:
import requests
import os
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup as bs
import csv
import time

### Creating coaches.csv

In [4]:
url = 'https://www.pro-football-reference.com/coaches/'
df = pd.read_html(url)[0]

In [5]:
df.tail()

,Rk,Coach,Yrs,From,To,G,W,L,T,W-L%,...,Yr plyf,G plyf,W plyf,L plyf,W-L%.1,AvRk,BstRk,Chmp,SBwl,Conf
526,527,Johnny Murphy,1,1924,1924,4,0,4,0,0.0,...,0.0,0.0,0.0,0.0,NaN,16.0,16,0.0,0.0,0.0
527,528,Al Nesser,1,1926,1926,2,0,1,1,0.0,...,0.0,0.0,0.0,0.0,NaN,16.0,16,0.0,0.0,0.0
528,529,Tam Rose,1,1921,1921,1,0,1,0,0.0,...,0.0,0.0,0.0,0.0,NaN,18.0,18,0.0,0.0,0.0
529,530,Lenny Sachs,1,1926,1926,4,0,4,0,0.0,...,0.0,0.0,0.0,0.0,NaN,21.0,21,0.0,0.0,0.0
530,531,Giff Smith,1,2023,2023,3,0,3,0,0.0,...,NaN,NaN,NaN,NaN,NaN,4.0,4,NaN,NaN,NaN


In [6]:
soup = bs(requests.get(url).content)

In [7]:
coaches = []
seen = set()
for tag in soup.find_all('a'):
    coach = tag['href'].split('/')[-1].split('.')[0]
    if '/coaches/' in tag['href'] and coach not in seen:
        coaches.append(coach)
        seen.add(coach)

In [8]:
def clean_hof(name):
    if name[-1] == '+' : return name[:-1]
    return name

In [9]:
key = pd.DataFrame(coaches,columns=['id']).iloc[:-1,:]
key['name'] = df['Coach'].apply(clean_hof)
key['to'] = df['To']
key.to_csv('input/coaches.csv')

### Creating photos.csv

In [10]:
from PIL import Image
coaches = pd.read_csv('input/coaches.csv',index_col=0)
coaches.head()

,id,name,to
0,ShulDo0,Don Shula,1995
1,HalaGe0,George Halas,1967
2,BeliBi0,Bill Belichick,2023
3,ReidAn0,Andy Reid,2024
4,LandTo0,Tom Landry,1988


In [14]:
coaches.iloc[3][0]

'ReidAn0'

In [11]:
def get_photo(coach,fetch=False):
    path = f'processed/img/{coach}.png'
    if not fetch and os.path.exists(path):
        print(f'path: {path} exists!')
        return
    else :
        base = 'https://www.pro-football-reference.com/coaches/'
        url = f'{base}/{coach}.htm'
        soup = bs(requests.get(url).content)
        valid = [img['src'] for img in soup.find_all('img') if 'head' in img['src']]
        img = valid[0]
        with open(path,'wb') as f:
            f.write(img)
        print(f'image scraped, now at path: {path}')
        return

In [ ]:
with open(f'processed/img/{id}.png','wb') as f:
    f.write(img)

In [107]:
# # this took forever
# tqdm.pandas()
# coaches['photo'] = coaches['id'].progress_map(get_photo_link)
# coaches.to_csv('photos.csv')

100%|██████████| 531/531 [5:58:00<00:00, 40.45s/it]    


In [152]:
coaches = pd.read_csv('photos.csv',index_col=0)
valid = coaches[coaches['photo'].notna()]

In [158]:
# for coach in tqdm(valid.iterrows()):
#     id = coach[1]['id']
#     p = coach[1]['photo']
#     img = requests.get(p).content
#     with open(f'processed/img/{id}.png','wb') as f:
#         f.write(img)
#     time.sleep(10)